<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **ACCESS to TEMPO data via OPeNDAP**

 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> EDL authentication (username/password)
2. <font size="3"> **Optional**: If running notebook locally, install the conda environment in `environment.yml` file and install conda environment to run notebook.


 <span style='color:#ff6666'><font size="5"> **Objectives**
### Basics
- <font size="3"> Brief Introduction to <span style='color:#ff6666'>**OPeNDAP**</span> (i.e. **dap2** vs **dap4**). 
- <font size="3"> How to find <span style='color:#ff6666'>**OPeNDAP**</span> URLs.
- <font size="3"> Inspecting metadata (differences between **browser** / <span style='color:#ff6666'>**pydap**</span> and **Xarray**).
- <font size="3"> **Naive approach**: access data from a url using **Xarray** + <span style='color:#ff6666'>**pydap**<span style='color:black'>. Here we demonstrate different ways to authenticate.

### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection

### Subset multiple remote files

- <font size="3"> **a)** Naive approaches.
- <font size="3"> **b)** Streaming data

## Appendix: Using curl


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess

# import pydap-specific tools
from pydap.net import create_session, extract_session_state
from pydap.client import get_cmr_urls, consolidate_metadata, stream, stream_parallel, open_url


# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**

In [ ]:
TEMPO_O3TOT_L3_V03_ccid = "C2930764281-LARC_CLOUD"
time_range = [dt.datetime(2025, 9, 1), dt.datetime(2025, 9, 30)] # One month of data

cmr_urls = get_cmr_urls(ccid=TEMPO_O3TOT_L3_V03_ccid, time_range=time_range, limit=1000) # you can incread the limit of results


print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[0]

# Inspecting Metadata

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Understanding DAP4**

<div style="text-align:center">
<img src="img/DAP4vsDAP2.png" alt="drawing" width="500"/>    
</div>


### Basic Data Exploration

<font size="3.5">To inspect the metadata of a remote OPeNDAP URL (variables in the file), you :

* <font size="3.5"> Append a `.dmr` at the end of the URL
* <font size="3.5"> Open on a browser.
* <font size="3.5"> Useful when interested in subseting by variables for example.

For example:

https://opendap.earthdata.nasa.gov/collections/C2930764281-LARC_CLOUD/granules/TEMPO_O3TOT_L3_V03_20250831T232841Z_S016.nc.dmr





<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via OPeNDAP**

<font size="3.5"> You can authenticate via:

* <font size="3.5"> `.netrc` file (username password)
* <font size="3.5"> Token bearer header


<font size="3.5"> OPeNDAP's Hyrax server support both forms of authentication. Below we demonstrate using earthaccess to store and inherit EDL credentials into a session that will be used to stream data from OPeNDAP in the Cloud.


In [ ]:
auth = earthaccess.login(strategy="interactive", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session = create_session(session=auth.get_session())

# Accessing Metadata

<font size="3.5"> What are some tools, their differences, and what do they do.

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **PYDAP: Metadata-Only**



```{note}
Q: How do we tell the server which protocol to use?
A: By replaing the http -> dap4 in the URL
```


In [ ]:
%%time
pyds = open_url(cmr_urls[0].replace("https","dap4"), session=my_session)

In [ ]:
pyds.tree()

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **XARRAY: PYDAP as Engine**


In [ ]:
%%time
dt = xr.open_datatree(cmr_urls[0].replace("https","dap4"), session=my_session, engine='pydap')

In [ ]:
dt


```{note}
Q: Why is it slower with Xarray + Pydap?
A: Xarray eagerly downloads all dimension data into memory before creating the Dataset Object
```


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **NAIVE APPROACHES when accessing OPeNDAP in the Cloud**

* <font size="3.5"> When aggregating multiple remote files with Xarray for data exploration
* <font size="3.5"> When downloading data into a file

### Solutions
* <font size="3.5"> Use pydap's logic to increase performance
* <font size="3.5"> Construct Constraint Expressions to reduce the metadata that Xarray parses

```{note}
While Xarray has a .drop_variables method to "drop" variables before say storing into a file, this dropping takes a place after creating the Xarray Dataset object. In some cases where a single granule has 1000 variables, this approach can be subperformant.
```


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **How to best Aggregate Multiple Files with Xarray**


<font size="3.5"> Below we demonstrate the performance of Xarray when aggregating multiple remote granules with OPeNDAP URLs (over https) with 2 approaches:

* <font size="3.5"> Naive approach.
* <font size="3.5"> Employing PyDAP's internal methods for "consolidating metadata".


<span style='font-family:serif'> <font size="4.5"><span style='color:#0066cc'> **Naive Approach**


In [ ]:
# Convert URLS into DAP4 urls
dap4_urls = [url.replace("https", 'dap4') for url in cmr_urls]


In [ ]:
%%time
ds = xr.open_mfdataset(dap4_urls, engine='pydap', session=my_session, parallel=True, concat_dim='time', combine='nested')

In [ ]:
ds

<span style='font-family:serif'> <font size="4.5"><span style='color:#0066cc'> **NOT-SO Naive Approach**

* <font size="3.5"> Consolidating Metadata via PYDAP.

<font size="3.5"> This approach generates a SQLite object on a local directory, storing all metadata for later reuse. THis is, can consolidate all metadata into a single file which pydap can natively use to speed up the Xarray Dataset object generation. We demonstrate below.


In [ ]:
metadata_name = "./data/TEMPO_O3TOT_L3_V03"
cache_kwargs = {'cache_name': metadata_name}
new_session = create_session(use_cache=True, session=auth.get_session(), cache_kwargs=cache_kwargs)
new_session

In [ ]:
new_session.cache.clear()

In [ ]:
%%time
consolidate_metadata(dap4_urls, session=new_session, concat_dim='time')

In [ ]:
%%time
ds1 = xr.open_mfdataset(dap4_urls, engine='pydap', session=new_session, parallel=True, concat_dim='time', combine='nested')
ds1

In [ ]:
%%time
ds2 = xr.open_mfdataset(dap4_urls, engine='pydap', session=new_session, parallel=True, concat_dim='time', combine='nested', group='/product')
ds3 = xr.open_mfdataset(dap4_urls, engine='pydap', session=new_session, parallel=True, concat_dim='time', combine='nested', group='/support_data')
ds4 = xr.open_mfdataset(dap4_urls, engine='pydap', session=new_session, parallel=True, concat_dim='time', combine='nested', group='/geolocation')
ds = xr.merge([ds1, ds2, ds3, ds4])
ds

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Server-Side Subsetting**

* <font size="3.5"> The DAP4 protocol supports server-side subseting by **a)** Variable name, and by **b)** Index space with the use of Constraint Expressions (additional query parameters appended to each URL).
* <font size="3.5"> Constraint Expressions (CEs) can be used to **a)** speed up metadata dataset object creation, and to download a subset of data (as opposed to downloading entire file and subsetting locally).


<font size="3.5"> **Below we demonstrate how to make sure the subset is done by the server and not by Xarray once the data has been downloaded**.
<font size="3.5"> Because xarray loads dimension (coordinates) into memory,for L3 data it is easy to identify a single spatial subset that applies to all granules.



In [ ]:
# define bounding box by edges
lat_min, lat_max = 45, 65
lon_min, lon_max = -123, -120.5

lat_index = dt.latitude.to_index()
lon_index = dt.longitude.to_index()

lat_start = lat_index.get_indexer([lat_min], method="nearest")[0]
lat_end   = lat_index.get_indexer([lat_max], method="nearest")[0]

lon_start = lon_index.get_indexer([lon_min], method="nearest")[0]
lon_end   = lon_index.get_indexer([lon_max], method="nearest")[0]


### Single granule case

In [ ]:
ds = xr.open_datatree(dap4_urls[0], engine='pydap', session=my_session)


In [ ]:
%%time
ds['support_data/terrain_height'].isel(latitude=slice(lat_start, lat_end), longitude=slice(lon_start, lon_end)).plot()

<span style='font-family:serif'> <font size="5"><span style='color:#0066cc'> **When slicing a Variable with Xarray (for a single granule)** </span> <font size="4.5"> Xarray passes down via pydap the request so that subsetting takes place close to the data. 


* <font size="3.5"> **When creating an aggregated view of the dataset, to ensure the server does the subsetting, the user must pass a chunk argument when creating the dataset.**

For example:
```python
ds1 = xr.open_mfdataset(
    dap4_urls, engine='pydap', 
    session=new_session, 
    parallel=True, 
    concat_dim='time', 
    combine='nested', 
    group='/',
    chunk={'latitude': size_of_lat_slice, 'longitude': size_of_lon_slice}
)

ds2 = xr.open_mfdataset(
    dap4_urls, engine='pydap', 
    session=new_session, 
    parallel=True, 
    concat_dim='time', 
    combine='nested', 
    group='/support_data',
    chunk={'latitude': size_of_lat_slice, 'longitude': size_of_lon_slice}
)

ds = xr.merge([ds1, ds2])
ds['support_data/terrain_height'].isel(longitude=slice(lon_start, lon_end), latitude=slice(lat_start,lat_end)).to_netcdf('local_file')
```

<font size="3.5"> <span style='color:#0066cc'>**If you do not chunk when creating the dataset, Xarray will download the ENTIRE variable into memory and then subset it, resulting in subpar performance and unnecessary data transfer (bug on Xarray).**</span>



<font size="3.5"> For reference, check this observation on the pydap documentation: 
* <font size="3.5"> [Subsetting 2 OPenDAP URLs](https://pydap.github.io/pydap/en/5_minute_tutorial.html#case-2-subsetting-across-two-separate-files), in particular the observation made in this block:
*  <font size="3.5"> [How to pass the slice from Xarray to the remote Server](https://pydap.github.io/pydap/en/5_minute_tutorial.html#how-to-pass-the-slice-from-xarray-to-the-remote-server)

<span style='font-family:serif'> <font size="5"><span style='color:#0066cc'> **Subsetting by Variable Names**

<font size="3.5"> This can be useful when the vast majory of variables will be discarded. In particular when the granule has O(100-1000) variables (typical of Level2 datasets). In that scenario, simply openning an Xarray object can take ~ 10-100 seconds or more for a single granule, since Xarray needs to parse all the metadata.

<font size="3.5"> A URL can be constructed that tells the OPeNDAP server which variables (and their dimensions) a user is interested. 





In [ ]:
keep_variables = ['/longitude', '/latitude', '/time', "/support_data", "/product/o3_below_cloud"]

CE = "?dap4.ce=" + ";".join(keep_variables)

dap4ce_urls = [url+CE for url in dap4_urls ]
dap4ce_urls[0]

In [ ]:
%%time
dt = xr.open_datatree(dap4ce_urls[0], engine='pydap', session=my_session)
dt

<span style='font-family:serif'> <font size="5"><span style='color:#0066cc'> **Recommended Worklow for Streaming Data**


* <font size="3.5"><span style='color:#0066cc'> **Data Exploration**</span>: `Xarray + PyDAP`. When interested in visualizing one or two variables. Identify regions of interest (interactively).

However, if the goal is to stream data via OPeNDAP, Xarray+opendap requires extra logic to get the **best performance** that is stil subpar compared to **talking to the server directly**


* <font size="3.5"><span style='color:#0066cc'> **Streaming a Data Subset**</span>: Pure PyDAP to parallelize direct server (subset) requests. See below:




In [ ]:
dim_slices = {'latitude': (lat_start, lat_end), 'longitude': (lon_start, lon_end)}

print('Varibles I want to stream into local file: \n\n', keep_variables, '\n\n ========================================================================') 

print("\npassing down to the server the following slice that will be applied to data variables of interest: \n\n", dim_slices,'\n\n ========================================================================')

In [ ]:
# This is necessary to pass down credentials since requests.Session is not a pickable object
session_state = session_state = extract_session_state(my_session)

In [ ]:
%%time
stream_parallel(
    cmr_urls[:4],
    session_state,
    keep_variables=keep_variables, 
    dim_slices=dim_slices,
    output_path="./data/",
    max_workers=4,
)

## check the data

In [ ]:
%%time
ds1 = xr.open_mfdataset("TEMPO_O3*.nc", group="/", parallel=True)
ds2 = xr.open_mfdataset("TEMPO_O3*.nc", group="/support_data", parallel=True, concat_dim='time', combine='nested')
ds3 = xr.open_mfdataset("TEMPO_O3*.nc", group="/product", parallel=True, concat_dim='time', combine='nested')
ds = xr.merge([ds1, ds2, ds3])

In [ ]:
ds